In [ ]:
!pip install dowhy
!pip install networkx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.5/245.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 95.1 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: cvxpy
    Found existing installation: cvxpy 1.6.7
    Uninstalling cvxpy-1.6.7:
      Successfully uninstalled cvxpy-1.6.7


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
project_path = "/content/drive/MyDrive/Colab Notebooks/CD Skripsi"

In [ ]:
import networkx as nx
from dowhy import gcm
import pandas as pd
import numpy as np

#DoWhy

In [ ]:
data_cluster3 = pd.read_csv(
    '/content/drive/MyDrive/Colab Notebooks/CD Skripsi/causal_cluster_3.csv',
    sep=',',
    engine='python'
)

data_cluster3.head()

,cancellation,no_delivery,delivery_delay,delivery_time,late_delivery_flag,avg_installments,total_payment,freight_value,product_description_length,product_photos_qty,is_dissatisfied
0,0.0,0.0,-17.5,6.5,0.0,2.0,354.37,36.48,631.5,1.0,0
1,0.0,0.0,-19.0,17.5,0.0,1.0,123.25,41.80,1022.0,1.0,0
2,0.0,0.0,-12.5,5.5,0.0,1.0,252.40,52.80,459.5,1.5,0
3,0.0,0.0,-10.0,13.0,0.0,1.5,214.90,32.00,593.0,6.5,0
4,0.0,0.0,-16.0,12.5,0.0,8.0,236.30,52.87,2267.5,3.0,0


In [ ]:
graph = nx.DiGraph()
graph.add_edges_from([
    ('delivery_delay', 'late_delivery_flag'),
        ('delivery_time', 'late_delivery_flag'),
        ('freight_value', 'delivery_time'),
        ('late_delivery_flag', 'is_dissatisfied'),
        ('avg_installments', 'total_payment'),
        ('product_description_length', 'total_payment'),
        ('delivery_delay', 'delivery_time'),
        ('total_payment', 'delivery_time'),
        ('freight_value', 'no_delivery'),
        ('freight_value', 'delivery_delay'),
        ('no_delivery', 'cancellation'),
        ('delivery_time', 'is_dissatisfied'),
        ('total_payment', 'freight_value'),
        ('avg_installments', 'delivery_time')
])

In [ ]:
causal_model = gcm.StructuralCausalModel(graph)

In [ ]:
gcm.auto.assign_causal_mechanisms(causal_model, data_cluster3)

In [ ]:
gcm.fit(causal_model, data_cluster3)

Fitting causal mechanism of node cancellation: 100%|██████████| 10/10 [00:00<00:00, 11.97it/s]


In [ ]:
validation_summary = gcm.evaluate_causal_model(causal_model, data_cluster3)
validation_summary

Test permutations of given graph: 100%|██████████| 50/50 [03:38<00:00,  4.37s/it]


CausalModelEvaluationResult(mechanism_performances={'avg_installments': MechanismPerformanceResult(), 'product_description_length': MechanismPerformanceResult(), 'total_payment': MechanismPerformanceResult(), 'freight_value': MechanismPerformanceResult(), 'no_delivery': MechanismPerformanceResult(), 'delivery_delay': MechanismPerformanceResult(), 'cancellation': MechanismPerformanceResult(), 'delivery_time': MechanismPerformanceResult(), 'late_delivery_flag': MechanismPerformanceResult(), 'is_dissatisfied': MechanismPerformanceResult()}, pnl_assumptions={'delivery_delay': (np.float64(1.0), np.False_, 0.05), 'late_delivery_flag': (np.float64(0.0), np.True_, 0.05), 'delivery_time': (np.float64(1.0), np.False_, 0.05), 'freight_value': (np.float64(0.8780755755692615), np.False_, 0.05), 'is_dissatisfied': (np.float64(0.0), np.True_, 0.05), 'total_payment': (np.float64(1.0), np.False_, 0.05), 'no_delivery': (np.float64(0.0), np.True_, 0.05), 'cancellation': (np.float64(0.0), np.True_, 0.05)}

In [ ]:
baseline=data_cluster3["is_dissatisfied"].mean()
baseline

np.float64(0.10311143270622286)

intervensi

In [ ]:
# INTERVENSI 1
# PERCEPAT PENGIRIMAN
# delivery time turun 10%
int_delivery = gcm.interventional_samples(
    causal_model,
    interventions={
        "delivery_time": lambda x: x * 0.9
    },
    observed_data = data_cluster3
)


In [ ]:
print("is_dissatisfied intervention:")
print(data_cluster3["is_dissatisfied"].mean())

is_dissatisfied intervention:
0.10311143270622286


In [ ]:
print("after intervention")
print(int_delivery["is_dissatisfied"].mean())

after intervention
0.09442836468885674


In [ ]:
comparison = pd.DataFrame({
    "Variabel": ["delivery_time", "is_dissatisfied"],
    "Sebelum": [
        data_cluster3["delivery_time"].mean(),
        data_cluster3["is_dissatisfied"].mean()
    ],
    "Sesudah": [
        int_delivery["delivery_time"].mean(),
        int_delivery["is_dissatisfied"].mean()
    ]
})

comparison["Efek Intervensi"] = comparison["Sesudah"] - comparison["Sebelum"]

comparison

,Variabel,Sebelum,Sesudah,Efek Intervensi
0,delivery_time,11.812454,10.692510,-1.119945
1,is_dissatisfied,0.103111,0.094428,-0.008683


In [ ]:
# INTERVENSI 2
# late_delivery_flag
int_late = gcm.interventional_samples(
    causal_model,
    interventions={
        "late_delivery_flag": lambda x: x * 0.9
    },
    observed_data = data_cluster3
)

In [ ]:
print("is_dissatisfied intervention:")
print(data_cluster3["is_dissatisfied"].mean())

is_dissatisfied intervention:
0.10311143270622286


In [ ]:
print("after intervention")
print(int_late["is_dissatisfied"].mean())

after intervention
0.11432706222865413


In [ ]:
comparison = pd.DataFrame({
    "Variabel": ["late_delivery_flag", "is_dissatisfied"],
    "Sebelum": [
        data_cluster3["late_delivery_flag"].mean(),
        data_cluster3["is_dissatisfied"].mean()
    ],
    "Sesudah": [
        int_late["late_delivery_flag"].mean(),
        int_late["is_dissatisfied"].mean()
    ]
})

comparison["Efek Intervensi"] = comparison["Sesudah"] - comparison["Sebelum"]

comparison

,Variabel,Sebelum,Sesudah,Efek Intervensi
0,late_delivery_flag,0.056523,0.051829,-0.004694
1,is_dissatisfied,0.103111,0.114327,0.011216


In [ ]:
# RINGKASAN
summary = pd.DataFrame({
    "Scenario": [
        "Baseline",
        "Delivery Time ↓",
        "Late Delivery Flag ↓",
    ],
    "Mean_is_dissatisfied": [
        baseline,
        int_delivery["is_dissatisfied"].mean(),
        int_late["is_dissatisfied"].mean(),
    ]
})

display(summary)

,Scenario,Mean_is_dissatisfied
0,Baseline,0.103111
1,Delivery Time ↓,0.094428
2,Late Delivery Flag ↓,0.114327
